In [1]:
import numpy as np
import pandas as pd
import torch
import pickle
from pathlib import Path

import MF_class as MF

np.random.seed(42)
if np.random.choice(np.arange(1000)) != 102:
    raise ValueError("Random seed is not set correctly.")

```
                                USERS                             
         ┌───────────────────────────────────────────────────────┐
         │                          │                            │
         │                          │                            │
         │                          │                            │
         │                          │                            │
ITEMS    │                          │                            │
         │                          │                            │
         │                          │                            │
         ├──────────────────────────┼────────────────────────────┤
         │                          │████████████████████████████│
         │                          │████████████████████████████│
         └───────────────────────────────────────────────────────┘
```

# 1. Choose Dataset

In [2]:
datasets = ['ml-1m', 'steam', 'goodreads']
DATASET = datasets[0]

base_artifacts = Path.cwd().resolve().parents[1] / 'CausalI2I_artifacts'
data_path = base_artifacts / 'Datasets' / 'Processed' / DATASET

In [ ]:
parameters_dict = {
    'ml-1m': {
        'n_factors': 40,
        'lr': 1e-3,
        'batch_size': 2**15,
        'n_epochs': 25},
    'steam': {
        'n_factors': 50,
        'lr': 2e-3,
        'batch_size': 2**16,
        'n_epochs': 30},
    'goodreads': {
        'n_factors': 50,
        'lr': 1e-3,
        'batch_size': 2**15,
        'n_epochs': 20}
}

n_factors  = parameters_dict[DATASET]['n_factors']
lr         = parameters_dict[DATASET]['lr']
batch_size = parameters_dict[DATASET]['batch_size']
n_epochs   = parameters_dict[DATASET]['n_epochs']

# 2. Load Data

In [4]:
train = pd.read_csv(data_path / 'train.csv')
test = pd.read_csv(data_path / 'test.csv')
with open(data_path / 'item_dict.pkl', 'rb') as f:
    item_dict = pickle.load(f)

n_users = train['user_id'].nunique()
n_items = train['item_id'].nunique()
print(f'Number of users: {n_users}, Number of items: {n_items}')

Number of users: 6040, Number of items: 3706


# 3. Train Model

In [ ]:
model = MF.MatrixFactorizationTorch(
    n_users=n_users, 
    n_items=n_items, 
    n_factors=n_factors
)

model.fit(
    train_data=train.values,
    val_data=test.values,
    lr=lr, 
    wd=1e-7,
    pos_weight=1,
    batch_size=batch_size,
    n_epochs=n_epochs,
    device=torch.device('cuda:0'), 
    use_amp=True)

Epoch  ||- - - - - - - - Train - - - - - - - -||- - - - - - Validation - - - - - - - || Epoch's | COS θ | Time     
Number || BCE    | BCE-POS | BCE-NEG | MPR    || BCE    | BCE-POS | BCE-NEG | MPR    || Change  |       | Elapsed  
=======||========|=========|=========|========||========|=========|=========|========||=========|=======|==========
   1   || 0.1369 |  2.0741 |  0.0461 | 0.8698 || 0.1394 |  2.0111 |  0.0536 | 0.8672 || 165.35  | None  | 00:03.91
   2   || 0.1252 |  1.9658 |  0.0389 | 0.8854 || 0.1261 |  1.9878 |  0.0407 | 0.8813 ||  42.09  | 0.663 | 00:07.90
   3   || 0.1154 |  1.7634 |  0.0381 | 0.8995 || 0.1172 |  1.8324 |  0.0386 | 0.8940 ||  42.20  | 0.713 | 00:11.81
   4   || 0.1077 |  1.6145 |  0.0371 | 0.9124 || 0.1108 |  1.7085 |  0.0375 | 0.9053 ||  36.57  | 0.783 | 00:15.79
   5   || 0.1026 |  1.5199 |  0.0362 | 0.9205 || 0.1064 |  1.6289 |  0.0366 | 0.9129 ||  27.86  | 0.880 | 00:19.77
   6   || 0.0988 |  1.4545 |  0.0352 | 0.9261 || 0.1032 |  1.5649 |  0.0362 |

### Save Model

In [22]:
name = f'MF_model_{DATASET}'

model_path = base_artifacts / 'Propensity_Models'
model.save(path=model_path / (name + '.pt'), note=None)

dict_out = {
    'n_users': n_users,
    'n_items': n_items,
    'n_factors': n_factors,
}

with open(model_path / f'MF_params_{DATASET}.pkl', 'wb') as f:
    pickle.dump(dict_out, f)

### Load Model

In [23]:
model_path = base_artifacts / 'Propensity_Models'
with open(model_path / f'MF_params_{DATASET}.pkl', 'rb') as f:
    loaded_params = pickle.load(f)

In [24]:
name = f'MF_model_{DATASET}'

loaded_model = MF.MatrixFactorizationTorch(
    n_users=loaded_params['n_users'], 
    n_items=loaded_params['n_items'], 
    n_factors=loaded_params['n_factors']
)

model_path = base_artifacts / 'Propensity_Models'
loaded_model.load(path=model_path / (name + '.pt'))

Loaded model summary:
Model:                      MatrixFactorizationTorch
Number of users:            6040
Number of items:            3706
Number of factors:          40
Learning rate:              0.001
Weight decay:               1e-07
Positive weight:            1
Batch size:                 32768
Number of epochs:           25
Device:                     cuda:0
Use AMP:                    True
Timestamp:                  2026-04-11 10:50:14
